# Chapter 10 — Semantic Networks

**The Encyclopedia of Artificial Intelligence** — Perla Genovesi

This notebook contains the Python example from Chapter 10. Press the play button (▶) next to the code cell to run it: the result appears just below.

No installation is needed. The code uses only Python's standard library.

In [ ]:
EDGES = [  # triples (node, relation, node)
    ("canary", "is-a", "bird"), ("ostrich", "is-a", "bird"),
    ("bird", "is-a", "animal"), ("fish", "is-a", "animal"),
    ("salmon", "is-a", "fish"),
    ("canary", "colour", "yellow"), ("canary", "can", "sing"),
    ("ostrich", "not-can", "fly"), ("bird", "can", "fly"),
    ("bird", "has", "wings"), ("bird", "has", "feathers"),
    ("fish", "can", "swim"), ("fish", "has", "gills"),
    ("animal", "can", "breathe"), ("animal", "has", "skin"),
]

def outgoing(node):
    return [(r, b) for a, r, b in EDGES if a == node]

def verify(subject, relation, value):
    """Climb the is-a edges: the closest information wins (exceptions)."""
    path, node = [subject], subject
    while node is not None:
        edges = outgoing(node)
        if ("not-" + relation, value) in edges:
            return False, path
        if (relation, value) in edges:
            return True, path
        node = next((b for r, b in edges if r == "is-a"), None)
        if node is not None:
            path.append(node)
    return None, path                      # open world: unknown

def activation(sources, steps=2, decay=0.5):
    """Spread activation along the edges, in both directions."""
    level = {s: 1.0 for s in sources}
    frontier = dict(level)
    for _ in range(steps):
        new = {}
        for node, a in frontier.items():
            for x, _, y in EDGES:
                if node in (x, y):
                    other = y if node == x else x
                    new[other] = new.get(other, 0) + a * decay
        for node, a in new.items():
            level[node] = level.get(node, 0) + a
        frontier = new
    return sorted(level.items(), key=lambda kv: -kv[1])

for s, r, v in [("canary", "can", "sing"), ("canary", "can", "fly"),
                ("canary", "has", "skin"), ("ostrich", "can", "fly"),
                ("salmon", "can", "fly")]:
    outcome, path = verify(s, r, v)
    print(f"{s} {r} {v}? {outcome}  (path: {' → '.join(path)}, "
          f"levels: {len(path) - 1})")
print("activation from canary:",
      [(n, round(a, 2)) for n, a in activation(["canary"])
       if n != "canary"])

## Expected output

If everything works, you should see:

```
canary can sing? True  (path: canary, levels: 0)
canary can fly? True  (path: canary → bird, levels: 1)
canary has skin? True  (path: canary → bird → animal, levels: 2)
ostrich can fly? False  (path: ostrich, levels: 0)
salmon can fly? None  (path: salmon → fish → animal, levels: 2)
```

Now try changing a number in the code and run it again to see what happens.